# 02 — Baseline CNN: Train-from-Scratch Floor Model


In [ ]:
import sys, os
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from src.utils import set_seed, get_device, plot_loss_curves, plot_confusion, compute_f1_report
from src.dataset import load_eurasat, IndexSubsetDataset, EUROSAT_CLASSES
from src.model import BaselineCNN
from src.train import run_training, evaluate

SEED = 42
set_seed(SEED)
device = get_device()
print("Device:", device)

DATA_ROOT = "../data/raw"
PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

eurosat = load_eurasat(root=DATA_ROOT, download=True)

manifest = np.load(os.path.join(PROCESSED_DIR, "split_manifest.npz"))
print("Manifest keys:", list(manifest.keys()))

ImportError: cannot import name 'IndexSubsetDataset' from 'src.dataset' (c:\Users\anshi\Celebal_Technology_Assignment\satellite-landuse-classifier\notebooks\..\src\dataset.py)

## 1. Transforms & DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

BATCH_SIZE = 64

def make_loaders(split_dict):
    train_ds = IndexSubsetDataset(eurosat, split_dict["train"], transform=train_transform)
    val_ds = IndexSubsetDataset(eurosat, split_dict["val"], transform=eval_transform)
    test_ds = IndexSubsetDataset(eurosat, split_dict["test"], transform=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader

naive_split_dict = {"train": manifest["naive_train"], "val": manifest["naive_val"], "test": manifest["naive_test"]}
block_split_dict = {"train": manifest["block_train"], "val": manifest["block_val"], "test": manifest["block_test"]}

naive_train_loader, naive_val_loader, naive_test_loader = make_loaders(naive_split_dict)
block_train_loader, block_val_loader, block_test_loader = make_loaders(block_split_dict)

print("Naive split batches -> train:", len(naive_train_loader), "val:", len(naive_val_loader), "test:", len(naive_test_loader))
print("Block split batches -> train:", len(block_train_loader), "val:", len(block_val_loader), "test:", len(block_test_loader))

NameError: name 'manifest' is not defined

## 2. Train BaselineCNN on the NAIVE random split

In [ ]:
set_seed(SEED) 

model_naive = BaselineCNN(num_classes=len(EUROSAT_CLASSES)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_naive = torch.optim.Adam(model_naive.parameters(), lr=0.001)

NUM_EPOCHS = 15

history_naive = run_training(
    model_naive, naive_train_loader, naive_val_loader,
    criterion, optimizer_naive, device,
    num_epochs=NUM_EPOCHS,
    checkpoint_path="../models/baseline_cnn_naive.pt",
)

NameError: name 'SEED' is not defined

## 3. Results — BaselineCNN on Naive Split

In [ ]:
fig = plot_loss_curves(history_naive, title="BaselineCNN — Naive Split")
plt.show()

model_naive.load_state_dict(torch.load("../models/baseline_cnn_naive.pt", map_location=device))
test_loss_naive, test_acc_naive, preds_naive, labels_naive = evaluate(
    model_naive, naive_test_loader, criterion, device
)
print(f"Naive split — Test loss: {test_loss_naive:.4f} | Test accuracy: {test_acc_naive:.4f}")

fig = plot_confusion(labels_naive, preds_naive, EUROSAT_CLASSES, title="BaselineCNN — Naive Split — Confusion Matrix")
plt.show()

per_class_f1_naive, macro_f1_naive, report_naive = compute_f1_report(labels_naive, preds_naive, EUROSAT_CLASSES)
print(report_naive)
print("Macro-F1 (naive split):", round(macro_f1_naive, 4))

NameError: name 'plot_loss_curves' is not defined

## 4. Train BaselineCNN on the BLOCK split

In [ ]:
set_seed(SEED)

model_block = BaselineCNN(num_classes=len(EUROSAT_CLASSES)).to(device)
optimizer_block = torch.optim.Adam(model_block.parameters(), lr=0.001)

history_block = run_training(
    model_block, block_train_loader, block_val_loader,
    criterion, optimizer_block, device,
    num_epochs=NUM_EPOCHS,
    checkpoint_path="../models/baseline_cnn_block.pt",
)